In [ ]:
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
INTERIMDIR = CONFIGS['filepaths']['interim']
DEPRECDIR  = f'{INTERIMDIR}/deprecated'
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']

In [ ]:
PROFILE_VARS = {
    't':         {'label':'$T$',               'units':'K'},
    'q':         {'label':'$q$',               'units':'kg/kg'},
    'rh':        {'label':'RH',                'units':'%'},
    'thetae':    {'label':'$\\theta_{e}$',     'units':'K'},
    'thetaestar':{'label':'$\\theta_{e}^*$',   'units':'K'}}

sigma = {}
pressure = {}
for varname in PROFILE_VARS:
    sigma[varname]    = xr.open_dataarray(f'{INTERIMDIR}/{varname}.nc',engine='h5netcdf')
    pressure[varname] = xr.open_dataarray(f'{DEPRECDIR}/{varname}.nc',engine='h5netcdf')

ps = xr.open_dataarray(f'{INTERIMDIR}/ps.nc',engine='h5netcdf')
print(f'Sigma variables have dims:    {list(sigma["t"].dims)}')
print(f'Pressure variables have dims: {list(pressure["t"].dims)}')

In [ ]:
# Vertical profile comparison at a single grid point and timestep
LAT,LON,TIDX = 15.0,75.0,0

fig,axs = pplt.subplots(nrows=len(PROFILE_VARS),ncols=2,refwidth=3,refheight=2,share=False)
axs.format(suptitle=f'Vertical profiles at ({LAT}\u00b0N, {LON}\u00b0E), timestep {TIDX}',
           collabels=['Pressure coordinates','Sigma coordinates'])

for i,(varname,info) in enumerate(PROFILE_VARS.items()):
    # Pressure profile
    pda  = pressure[varname].sel(lat=LAT,lon=LON,method='nearest').isel(time=TIDX).load()
    levs = pda.lev.values
    axs[i,0].plot(pda.values,levs,color='blue3',linewidth=1.5,marker='o',markersize=3)
    axs[i,0].format(ylabel='Pressure (hPa)',xlabel=f'{info["label"]} ({info["units"]})',
                    yreverse=True,ylim=(1000,500))

    # Sigma profile
    sda  = sigma[varname].sel(lat=LAT,lon=LON,method='nearest').isel(time=TIDX).load()
    sigs = sda.sig.values
    axs[i,1].plot(sda.values,sigs,color='red6',linewidth=1.5,marker='o',markersize=3)
    axs[i,1].format(ylabel='$\\sigma$',xlabel=f'{info["label"]} ({info["units"]})',
                    yreverse=True,ylim=(1.0,0.5))

pplt.show()

In [ ]:
# Domain-mean vertical structure comparison
fig,axs = pplt.subplots(nrows=1,ncols=len(PROFILE_VARS),refwidth=2,refheight=3,share=False)
axs.format(suptitle='Domain-mean vertical structure')

for i,(varname,info) in enumerate(PROFILE_VARS.items()):
    # Pressure: mean over lat, lon, time
    pmean = pressure[varname].mean(dim=('lat','lon','time')).load()
    levs  = pmean.lev.values
    axs[i].plot(pmean.values,levs,color='blue3',linewidth=1.5,label='Pressure')
    axs[i].format(ylabel='Pressure (hPa)',xlabel=f'{info["label"]}\n({info["units"]})',
                  yreverse=True,ylim=(1000,500),title=info['label'])

    # Sigma: mean over lat, lon, time (secondary y-axis via twin)
    smean = sigma[varname].mean(dim=('lat','lon','time')).load()
    sigs  = smean.sig.values
    ax2   = axs[i].twinx()
    ax2.plot(smean.values,sigs,color='red6',linewidth=1.5,linestyle='--',label='Sigma')
    ax2.format(ylabel='$\\sigma$',yreverse=True,ylim=(1.0,0.5))

axs[0].legend(loc='lr',ncols=1)
pplt.show()

In [ ]:
# Spatial map comparison at a selected level (~850 hPa / sigma=0.85)
PLEV  = 850.0
SIGLEV = 0.85

fig,axs = pplt.subplots(nrows=len(PROFILE_VARS),ncols=2,proj='cyl',refwidth=3,refheight=2)
axs.format(coast=True,grid=False,latlim=LATRANGE,lonlim=LONRANGE,
           suptitle=f'Time-mean fields: pressure ({PLEV:.0f} hPa) vs sigma ({SIGLEV})',
           collabels=[f'Pressure ({PLEV:.0f} hPa)',f'Sigma ({SIGLEV})'])

for i,(varname,info) in enumerate(PROFILE_VARS.items()):
    pmap = pressure[varname].sel(lev=PLEV,method='nearest').mean('time').load()
    smap = sigma[varname].sel(sig=SIGLEV,method='nearest').mean('time').load()
    vmin = min(float(pmap.min()),float(smap.min()))
    vmax = max(float(pmap.max()),float(smap.max()))
    style = dict(vmin=vmin,vmax=vmax,cmap='Spectral_r')
    im = axs[i,0].pcolormesh(pmap.lon.values,pmap.lat.values,pmap.values,**style)
    axs[i,1].pcolormesh(smap.lon.values,smap.lat.values,smap.values,**style)
    axs[i,0].format(title=info['label'])
    fig.colorbar(im,loc='r',rows=i+1,label=info['units'])

pplt.show()